# Taiwan Stock AI — System Performance Diagnostic

## TL;DR

The local evidence is useful for diagnosing the measurement system, but it is not yet a professional backtest. It contains only five distinct signal dates, combines several holding horizons, includes repeated dry/manual runs, and applies an entry price that changes with the run clock and calendar. The core candidate stream is materially stronger than the small/mid-cap radar in this sample, but neither should be treated as proven without a fixed execution rule and a larger out-of-sample live cohort.

This notebook is deliberately read-only. It loads `signal_history.jsonl` and `signal_performance.jsonl`, recomputes all results from source rows, and does not call external market APIs.

## Context & Methods

- **Question:** Is the perceived mediocre win rate caused by weak signals, weak measurement, or both?
- **Primary analysis grain:** one signal date × code × source × horizon. This removes duplicate runs on the same date while retaining separate horizons.
- **Comparison grain:** raw rows and the application's current “last 200 rows” window are also shown to expose metric sensitivity.
- **Uncertainty:** confidence intervals use a cluster bootstrap by signal date because rows from the same date are not independent.
- **Cost sensitivity:** 0.6 percentage points round trip is an illustrative scenario only, not a statement of the user's broker fee.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

root = Path.cwd()
if not (root / "signal_history.jsonl").exists():
    root = root.parent

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

history = read_jsonl(root / "signal_history.jsonl")
performance = read_jsonl(root / "signal_performance.jsonl")

for frame in (history, performance):
    frame["source_norm"] = frame.get("source", pd.Series(index=frame.index, dtype="object")).fillna("core")
    frame["date"] = pd.to_datetime(frame["date"]).dt.date

history["created_at"] = pd.to_datetime(history["created_at"])
performance["evaluated_at"] = pd.to_datetime(performance["evaluated_at"])

print(f"history rows={len(history):,}; performance rows={len(performance):,}")
print(f"signal date range={history['date'].min()} to {history['date'].max()}")

history rows=83; performance rows=316
signal date range=2026-05-04 to 2026-06-13


## Data quality and analysis grain

In [2]:
history_run_key = ["run_id", "code", "source_norm"]
history_date_key = ["date", "code", "source_norm"]
performance_run_key = ["run_id", "code", "source_norm", "horizon"]
performance_date_key = ["date", "code", "source_norm", "horizon"]

quality = pd.DataFrame([
    {"check": "History rows", "value": len(history), "interpretation": "Rows written by all runs"},
    {"check": "Distinct run-level signals", "value": len(history.drop_duplicates(history_run_key)), "interpretation": "Exact run duplicates removed"},
    {"check": "Distinct date-level signals", "value": len(history.drop_duplicates(history_date_key)), "interpretation": "Recommended analysis grain"},
    {"check": "Distinct signal dates", "value": history["date"].nunique(), "interpretation": "Independent market-day clusters"},
    {"check": "Performance rows", "value": len(performance), "interpretation": "One row per recorded horizon"},
    {"check": "Missing source labels", "value": int(performance.get("source", pd.Series(index=performance.index)).isna().sum()), "interpretation": "Normalized to core for compatibility"},
    {"check": "POST-mode history share", "value": f"{(history['mode'].eq('POST').mean() * 100):.1f}%", "interpretation": "No local PRE cohort"},
])
display(quality)

run_profile = (history.groupby(["date", "run_id"], as_index=False)
               .agg(signals=("code", "size"), created_at=("created_at", "min")))
display(run_profile.sort_values(["date", "created_at"]))

,check,value,interpretation
0,History rows,83,Rows written by all runs
1,Distinct run-level signals,79,Exact run duplicates removed
2,Distinct date-level signals,64,Recommended analysis grain
3,Distinct signal dates,5,Independent market-day clusters
4,Performance rows,316,One row per recorded horizon
5,Missing source labels,120,Normalized to core for compatibility
6,POST-mode history share,100.0%,No local PRE cohort


,date,run_id,signals,created_at
0,2026-05-04,2026-05-04:POST:2026-05-04 22:02:49,3,2026-05-04 22:02:49
1,2026-05-04,2026-05-04:POST:2026-05-04 22:05:05,8,2026-05-04 22:05:05
2,2026-05-05,2026-05-05:POST:2026-05-05 00:52:29,8,2026-05-05 00:52:29
3,2026-05-05,2026-05-05:POST:2026-05-05 00:52:59,3,2026-05-05 00:52:59
4,2026-05-05,2026-05-05:POST:2026-05-05 00:53:56,8,2026-05-05 00:53:56
5,2026-06-03,2026-06-03:POST:2026-06-03 23:41:01,8,2026-06-03 23:41:01
6,2026-06-04,2026-06-04:POST:2026-06-04 00:14:47,13,2026-06-04 00:14:47
7,2026-06-04,2026-06-04:POST:2026-06-04 23:52:10,16,2026-06-04 23:52:10
8,2026-06-13,2026-06-13:POST:2026-06-13 18:46:30,16,2026-06-13 18:46:30


## Results: the headline changes with the row-selection rule

The application currently takes the final 200 performance rows in file order. Because each signal normally creates four horizon rows and evaluation writes do not necessarily follow signal chronology, this is not the same thing as the latest 200 independent signals.

In [3]:
performance = performance.sort_index().copy()
raw = performance.drop_duplicates(performance_run_key, keep="last")
date_dedup = (performance.sort_values("evaluated_at")
              .drop_duplicates(performance_date_key, keep="last"))
last_200 = performance.tail(200).copy()

def summarize(frame: pd.DataFrame, label: str) -> dict:
    wins = frame["return_pct"] > 0
    positive = frame.loc[wins, "return_pct"].sum()
    negative = -frame.loc[~wins, "return_pct"].sum()
    return {
        "view": label,
        "rows": len(frame),
        "signal_dates": frame["date"].nunique(),
        "mean_return_pct": frame["return_pct"].mean(),
        "median_return_pct": frame["return_pct"].median(),
        "win_rate_pct": wins.mean() * 100,
        "mean_excess_pct": frame["excess_return_pct"].mean(),
        "excess_win_rate_pct": (frame["excess_return_pct"] > 0).mean() * 100,
        "profit_factor": positive / negative if negative else np.nan,
    }

views = pd.DataFrame([
    summarize(last_200, "Application: last 200 file rows"),
    summarize(raw, "Run-level deduplicated rows"),
    summarize(date_dedup, "Date/code/source deduplicated rows"),
]).round(3)
display(views)

def by_horizon(frame: pd.DataFrame, view: str) -> pd.DataFrame:
    rows = []
    for horizon, group in frame.groupby("horizon"):
        row = summarize(group, view)
        row["horizon"] = int(horizon)
        rows.append(row)
    return pd.DataFrame(rows)[[
        "view", "horizon", "rows", "signal_dates", "mean_return_pct",
        "median_return_pct", "win_rate_pct", "mean_excess_pct", "excess_win_rate_pct"
    ]]

horizon_comparison = pd.concat([
    by_horizon(last_200, "last 200"),
    by_horizon(date_dedup, "date-level dedup"),
], ignore_index=True).round(3)
display(horizon_comparison)

,view,rows,signal_dates,mean_return_pct,median_return_pct,win_rate_pct,mean_excess_pct,excess_win_rate_pct,profit_factor
0,Application: last 200 file rows,200,4,-2.641,-2.096,35.000,-1.634,36.500,0.453
1,Run-level deduplicated rows,316,5,0.279,-0.635,43.987,0.454,40.823,1.076
2,Date/code/source deduplicated rows,256,5,0.478,-0.522,44.922,0.727,43.359,1.137


,view,horizon,rows,signal_dates,mean_return_pct,median_return_pct,win_rate_pct,mean_excess_pct,excess_win_rate_pct
0,last 200,1,50,4,-2.539,-1.746,20.000,-1.825,32.000
1,last 200,3,50,4,-3.766,-3.354,30.000,-2.297,32.000
2,last 200,5,50,4,-4.183,-2.702,38.000,-1.187,44.000
3,last 200,10,50,4,-0.075,0.331,52.000,-1.226,38.000
4,date-level dedup,1,64,5,-0.738,-1.074,32.812,-0.436,37.500
5,date-level dedup,3,64,5,-0.801,-1.289,42.188,-0.579,37.500
6,date-level dedup,5,64,5,0.623,0.167,51.562,1.726,53.125
7,date-level dedup,10,64,5,2.828,0.529,53.125,2.196,45.312


## Results: core and small/mid-cap radar are different products

In [4]:
source_rows = []
for (source, horizon), group in date_dedup.groupby(["source_norm", "horizon"]):
    source_rows.append({
        "source": source,
        "horizon": int(horizon),
        "rows": len(group),
        "signal_dates": group["date"].nunique(),
        "mean_return_pct": group["return_pct"].mean(),
        "win_rate_pct": (group["return_pct"] > 0).mean() * 100,
        "mean_excess_pct": group["excess_return_pct"].mean(),
        "excess_win_rate_pct": (group["excess_return_pct"] > 0).mean() * 100,
    })
source_by_horizon = pd.DataFrame(source_rows).round(3)
display(source_by_horizon)

source_overall = []
for source, group in date_dedup.groupby("source_norm"):
    result = summarize(group, source)
    result["unique_signals"] = len(group.drop_duplicates(["date", "code", "source_norm"]))
    source_overall.append(result)
source_overall = pd.DataFrame(source_overall).round(3)
display(source_overall)

recent_source = (last_200.groupby("source_norm", as_index=False)
                 .agg(rows=("return_pct", "size"),
                      mean_return_pct=("return_pct", "mean"),
                      win_rate_pct=("return_pct", lambda s: (s > 0).mean() * 100),
                      return_sum_pct=("return_pct", "sum")))
total_net_loss = -last_200["return_pct"].sum()
small_mid_net_loss = -last_200.loc[last_200["source_norm"] == "small_mid_radar", "return_pct"].sum()
print(f"Small/mid radar share of aggregate net loss in last-200 view: {small_mid_net_loss / total_net_loss * 100:.1f}%")
display(recent_source.round(3))

,source,horizon,rows,signal_dates,mean_return_pct,win_rate_pct,mean_excess_pct,excess_win_rate_pct
0,core,1,45,5,0.159,42.222,0.425,48.889
1,core,3,45,5,0.542,51.111,0.754,44.444
2,core,5,45,5,2.874,60.000,3.756,60.000
3,core,10,45,5,5.626,57.778,5.434,55.556
4,small_mid_radar,1,19,2,-2.861,10.526,-2.475,10.526
5,small_mid_radar,3,19,2,-3.980,21.053,-3.738,21.053
6,small_mid_radar,5,19,2,-4.707,31.579,-3.083,36.842
7,small_mid_radar,10,19,2,-3.798,42.105,-5.473,21.053


,view,rows,signal_dates,mean_return_pct,median_return_pct,win_rate_pct,mean_excess_pct,excess_win_rate_pct,profit_factor,unique_signals
0,core,180,5,2.300,0.701,52.778,2.593,52.222,1.763,45
1,small_mid_radar,76,2,-3.837,-2.386,26.316,-3.692,22.368,0.169,19


Small/mid radar share of aggregate net loss in last-200 view: 80.3%


,source_norm,rows,mean_return_pct,win_rate_pct,return_sum_pct
0,core,116,-0.898,43.103,-104.213
1,small_mid_radar,84,-5.047,23.810,-423.926


## Uncertainty: cluster bootstrap by signal date

Rows from one day share the same market regime and are correlated. Resampling individual rows would therefore overstate confidence. The bootstrap below resamples the five observed signal dates as clusters and reports percentile intervals.

In [5]:
rng = np.random.default_rng(20260716)
bootstrap_rows = []

for horizon, horizon_frame in date_dedup.groupby("horizon"):
    daily = (horizon_frame.groupby("date", as_index=False)
             .agg(n=("return_pct", "size"),
                  return_sum=("return_pct", "sum"),
                  excess_sum=("excess_return_pct", "sum"),
                  wins=("return_pct", lambda s: int((s > 0).sum()))))
    draws = rng.integers(0, len(daily), size=(20_000, len(daily)))
    n = daily["n"].to_numpy()[draws].sum(axis=1)
    mean_return = daily["return_sum"].to_numpy()[draws].sum(axis=1) / n
    mean_excess = daily["excess_sum"].to_numpy()[draws].sum(axis=1) / n
    win_rate = daily["wins"].to_numpy()[draws].sum(axis=1) / n * 100
    bootstrap_rows.append({
        "horizon": int(horizon),
        "signal_dates": len(daily),
        "mean_return_pct": horizon_frame["return_pct"].mean(),
        "return_ci_low": np.quantile(mean_return, 0.025),
        "return_ci_high": np.quantile(mean_return, 0.975),
        "win_rate_pct": (horizon_frame["return_pct"] > 0).mean() * 100,
        "win_ci_low": np.quantile(win_rate, 0.025),
        "win_ci_high": np.quantile(win_rate, 0.975),
        "mean_excess_pct": horizon_frame["excess_return_pct"].mean(),
        "excess_ci_low": np.quantile(mean_excess, 0.025),
        "excess_ci_high": np.quantile(mean_excess, 0.975),
    })

bootstrap = pd.DataFrame(bootstrap_rows).round(3)
display(bootstrap)

,horizon,signal_dates,mean_return_pct,return_ci_low,return_ci_high,win_rate_pct,win_ci_low,win_ci_high,mean_excess_pct,excess_ci_low,excess_ci_high
0,1,5,-0.738,-2.774,1.814,32.812,15.476,53.704,-0.436,-1.650,1.413
1,3,5,-0.801,-4.877,4.173,42.188,21.429,67.241,-0.579,-2.615,1.794
2,5,5,0.623,-5.700,8.602,51.562,23.810,84.483,1.726,-1.113,6.222
3,10,5,2.828,-0.408,7.419,53.125,44.286,65.000,2.196,-1.613,7.144


## Execution consistency: recorded signal price vs evaluated entry close

In [6]:
history_unique = (history.sort_values("created_at")
                  .drop_duplicates(history_run_key, keep="last"))
h1 = raw.loc[raw["horizon"] == 1].drop_duplicates(performance_run_key, keep="last")
entry = history_unique.merge(
    h1[["run_id", "code", "source_norm", "entry_close"]],
    on=["run_id", "code", "source_norm"],
    how="inner",
)
entry["entry_gap_pct"] = (entry["entry_close"] / entry["price"] - 1) * 100
entry["same_close_1bp"] = entry["entry_gap_pct"].abs() <= 0.01
entry["is_weekend"] = pd.to_datetime(entry["date"].astype(str)).dt.dayofweek >= 5

entry_summary = pd.DataFrame([{
    "matched_signals": len(entry),
    "same_close_within_1bp": int(entry["same_close_1bp"].sum()),
    "same_close_share_pct": entry["same_close_1bp"].mean() * 100,
    "mean_entry_gap_pct": entry["entry_gap_pct"].mean(),
    "min_entry_gap_pct": entry["entry_gap_pct"].min(),
    "max_entry_gap_pct": entry["entry_gap_pct"].max(),
    "weekend_signals": int(entry["is_weekend"].sum()),
}]).round(3)
display(entry_summary)

entry_by_run = (entry.groupby("run_id", as_index=False)
                .agg(signals=("code", "size"),
                     mean_entry_gap_pct=("entry_gap_pct", "mean"),
                     same_close_share_pct=("same_close_1bp", lambda s: s.mean() * 100),
                     weekend_signals=("is_weekend", "sum")))
display(entry_by_run.round(3))

,matched_signals,same_close_within_1bp,same_close_share_pct,mean_entry_gap_pct,min_entry_gap_pct,max_entry_gap_pct,weekend_signals
0,79,34,43.038,2.298,-5.84,20.881,14


,run_id,signals,mean_entry_gap_pct,same_close_share_pct,weekend_signals
0,2026-05-04:POST:2026-05-04 22:02:49,3,0.000,100.000,0
1,2026-05-04:POST:2026-05-04 22:05:05,8,0.000,100.000,0
2,2026-05-05:POST:2026-05-05 00:52:29,8,4.820,0.000,0
3,2026-05-05:POST:2026-05-05 00:52:59,3,11.793,0.000,0
4,2026-05-05:POST:2026-05-05 00:53:56,8,4.820,0.000,0
5,2026-06-03:POST:2026-06-03 23:41:01,8,0.000,100.000,0
6,2026-06-04:POST:2026-06-04 00:14:47,13,3.104,0.000,0
7,2026-06-04:POST:2026-06-04 23:52:10,14,0.000,100.000,0
8,2026-06-13:POST:2026-06-13 18:46:30,14,2.051,7.143,14


## Score discrimination and transaction-cost sensitivity

In [7]:
score_rows = history_unique[["run_id", "code", "source_norm", "score"]].merge(
    raw[["run_id", "code", "source_norm", "horizon", "return_pct"]],
    on=["run_id", "code", "source_norm"],
    how="inner",
)
core_scores = score_rows[score_rows["source_norm"] == "core"]
score_quality = []
for horizon, group in core_scores.groupby("horizon"):
    score_quality.append({
        "horizon": int(horizon),
        "rows": len(group),
        "unique_scores": group["score"].nunique(),
        "score_100_share_pct": (group["score"] == 100).mean() * 100,
        "spearman_score_vs_return": group["score"].rank(method="average").corr(
            group["return_pct"].rank(method="average")
        ),
    })
display(pd.DataFrame(score_quality).round(3))

cost_pp = 0.6
cost_sensitivity = pd.DataFrame([
    {
        "view": "run-level rows",
        "gross_mean_pct": raw["return_pct"].mean(),
        "net_mean_pct_at_0_6pp": (raw["return_pct"] - cost_pp).mean(),
        "gross_win_rate_pct": (raw["return_pct"] > 0).mean() * 100,
        "net_win_rate_pct_at_0_6pp": (raw["return_pct"] > cost_pp).mean() * 100,
    },
    {
        "view": "date-level dedup rows",
        "gross_mean_pct": date_dedup["return_pct"].mean(),
        "net_mean_pct_at_0_6pp": (date_dedup["return_pct"] - cost_pp).mean(),
        "gross_win_rate_pct": (date_dedup["return_pct"] > 0).mean() * 100,
        "net_win_rate_pct_at_0_6pp": (date_dedup["return_pct"] > cost_pp).mean() * 100,
    },
]).round(3)
display(cost_sensitivity)

,horizon,rows,unique_scores,score_100_share_pct,spearman_score_vs_return
0,1,58,9,22.414,-0.134
1,3,58,9,22.414,-0.255
2,5,58,9,22.414,-0.297
3,10,58,9,22.414,-0.156


,view,gross_mean_pct,net_mean_pct_at_0_6pp,gross_win_rate_pct,net_win_rate_pct_at_0_6pp
0,run-level rows,0.279,-0.321,43.987,40.823
1,date-level dedup rows,0.478,-0.122,44.922,41.797


## Takeaways

1. **Do not optimize the selector against the current headline win rate.** First fix the measurement contract: one immutable signal ID, explicit live/research environment, strategy version, fixed next-tradable-price entry, transaction costs, and chronological cohorts.
2. **Separate the core list and small/mid-cap radar.** The local sample says they have opposite behavior, and the radar has only two signal dates. Keep it in shadow mode until it passes an independent readiness gate.
3. **Use net excess expectancy as the primary outcome.** Keep win rate secondary and always label the horizon, sample size, signal dates, confidence interval, execution rule, and strategy version.
4. **Replace points with a calibrated ranking baseline only after measurement is fixed.** Use walk-forward splits, probability calibration, and ablation tests for technical, fundamental, and news feature families.
5. **Interpret this as a product-quality audit, not a trading recommendation.** The data is local, stale relative to the audit date, and may not match the Raspberry Pi production state.